In [1]:
# Manipulación de datos
import os
import json
import numpy as np
import pandas as pd
from math import pi
from datetime import timedelta

# Modelado y ML
from xgboost import XGBRegressor
from arch import arch_model

# Series de tiempo y descomposición
from statsmodels.tsa.seasonal import seasonal_decompose

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# Estadística (si usas quantiles t en GARCH)
from scipy.stats import t as student_t

# Gráficos
import matplotlib.pyplot as plt

# Opcional pero recomendado (evitar warnings incómodos)
import warnings
warnings.filterwarnings("ignore")


from Tools import Tools4DataExtraction as T4DE

In [2]:
D4NN= T4DE.Get_Data4NN_p()
atributo='FECHA_CORRIDA'
atributo='FECHA_OPERACION'
D4NN[atributo] = pd.to_datetime(D4NN[atributo])
fecha_maxima = D4NN[atributo].max()
dia_anterior = fecha_maxima - timedelta(days=1)
"""
fecha_inicio = dia_anterior - timedelta(days=364)
D4NN = D4NN[(D4NN[atributo] >= fecha_inicio) &
                    (D4NN[atributo] <= dia_anterior)].copy()
"""
D4NN = D4NN[(D4NN[atributo] <= dia_anterior)].copy()

#D4NN= D4NN[D4NN['TIPO_PASAJERO']=='AD'].copy()

Memoria usada antes: 630.44 MB
Memoria usada después: 122.47 MB
Reducción: 80.6%


In [3]:
# forecast_pipeline_adapted.py
# Versión: primer programa reestructurado a la forma del segundo (STL + modular)
import os
import json
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from arch import arch_model
from scipy.stats import t

# =====================================================
# 1. DATA PREPARATION
# =====================================================

def load_and_aggregate(D4NN, target_col):
    """
    Igual que el primer programa: agrupa por FECHA_CORRIDA y mantiene columnas originales.
    """
    Frame = D4NN[['FECHA_OPERACION', target_col, 'MES', 'PORCENT_PROMO', 'FOP']].copy()
    agg_dict = {col: 'mean' for col in Frame.columns if col not in ['FECHA_OPERACION', target_col]}
    agg_dict[target_col] = 'sum'

    df = Frame.groupby('FECHA_OPERACION').agg(agg_dict)
    df.index = pd.to_datetime(df.index)
    return df.sort_index()


def add_seasonal_and_calendar(df, target_col, period=14):
    """
    Reemplaza seasonal_decompose por STL (robusto) pero mantiene nombres de columnas del primer programa.
    """
    stl = STL(df[target_col], period=period, robust=True)
    result = stl.fit()
    df['trend'] = result.trend

    df['DiaSemana'] = df.index.dayofweek

    # Mantener columna MES que viene en df
    df["sin_mes"] = np.sin(2 * np.pi * df["MES"] / 12)
    df["cos_mes"] = np.cos(2 * np.pi * df["MES"] / 12)
    df["sin_dia"] = np.sin(2 * np.pi * df["DiaSemana"] / 7)
    df["cos_dia"] = np.cos(2 * np.pi * df["DiaSemana"] / 7)

    return df, result


def add_business_features(df, json_path):
    """
    Igual que primer programa: lee JSON con DiaBueno y MesBueno.
    """
    with open(json_path, 'r') as f:
        BC = json.load(f)
    df['DiaMayorBoletaje'] = df['DiaSemana'].isin(BC["DiaBueno"])
    df['MesMayorBoletaje'] = df['MES'].isin(BC["MesBueno"])
    return df, BC


def add_diff_features(df):
    """
    Mantener exactamente las diferencias y lags del primer programa.
    """
    df['trend_diff'] = df['trend'].diff()
    df["diff_lag1"] = df["trend_diff"].shift(1)
    df["diff_lag7"] = df["trend_diff"].shift(7)
    df["diff_lag30"] = df["trend_diff"].shift(30)

    df["diff_ma7"] = df["trend_diff"].rolling(7).mean()
    df["diff_ma30"] = df["trend_diff"].rolling(30).mean()
    return df.dropna()


# =====================================================
# 2. TRAINING
# =====================================================

def build_train_test(df, feature_cols, target="trend_diff", test_ratio=0.2):
    X = df[feature_cols]
    y = df[target]
    split = int(len(df) * (1 - test_ratio))
    return X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:], split


def train_xgboost(X_train, y_train):
    """
    Hiperparámetros del primer programa (800 trees, lr 0.03, depth 6, subsample 0.8).
    """
    model = XGBRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        verbosity=0,
        objective="reg:squarederror"
    )
    model.fit(X_train, y_train)
    return model


def reconstruct_trend(df, y_pred, split):
    last = df['trend'].iloc[split - 1]
    trend_pred = np.cumsum(y_pred) + last
    trend_real = df["trend"].iloc[split:].values
    return trend_real, trend_pred


# =====================================================
# 3. FUTURE FORECAST
# =====================================================

def forecast_autoregressive(df, model, BC_json, days=15):
    """
    Forecast autoregresivo idéntico al del primer programa, respetando nombres de columnas.
    """
    last_date = df.index[-1]
    future = pd.DataFrame(index=pd.date_range(last_date + pd.Timedelta(1), periods=days))

    # calendar
    future["MES"] = future.index.month
    future["DiaSemana"] = future.index.dayofweek
    future["sin_mes"] = np.sin(2 * np.pi * future["MES"] / 12)
    future["cos_mes"] = np.cos(2 * np.pi * future["MES"] / 12)
    future["sin_dia"] = np.sin(2 * np.pi * future["DiaSemana"] / 7)
    future["cos_dia"] = np.cos(2 * np.pi * future["DiaSemana"] / 7)

    # business rules
    future["DiaMayorBoletaje"] = future["DiaSemana"].isin(BC_json["DiaBueno"])
    future["MesMayorBoletaje"] = future["MES"].isin(BC_json["MesBueno"])

    # exogenas (naive)
    future["PORCENT_PROMO"] = df["PORCENT_PROMO"].iloc[-1]
    future["FOP"] = df["FOP"].iloc[-1]

    # autoregressive loop
    diffs = []
    last_trend = df["trend"].iloc[-1]
    last_diff = df["trend_diff"].iloc[-1]

    for i in range(days):
        row = pd.DataFrame({
            "diff_lag1": [last_diff],
            "diff_lag7": [df["trend_diff"].iloc[-7 + i] if i < 7 else diffs[i - 7]],
            "diff_lag30": [df["trend_diff"].iloc[-30 + i] if i < 30 else diffs[i - 30]],
            "diff_ma7": [df["trend_diff"].iloc[-7:].mean()],
            "diff_ma30": [df["trend_diff"].iloc[-30:].mean()],
            "sin_dia": [future["sin_dia"].iloc[i]],
            "cos_dia": [future["cos_dia"].iloc[i]],
            "sin_mes": [future["sin_mes"].iloc[i]],
            "cos_mes": [future["cos_mes"].iloc[i]],
            "DiaMayorBoletaje": [future["DiaMayorBoletaje"].iloc[i]],
            "MesMayorBoletaje": [future["MesMayorBoletaje"].iloc[i]],
            "PORCENT_PROMO": [future["PORCENT_PROMO"].iloc[i]],
            "FOP": [future["FOP"].iloc[i]],
        })

        pred = model.predict(row)[0]
        diffs.append(pred)
        last_trend += pred
        last_diff = pred

    future["pred_trend"] = np.cumsum(diffs) + df["trend"].iloc[-1]
    return future


def apply_garch_intervals(resid, future_df, seasonal, alpha=0.99):
    """
    Igual que primer programa: GARCH(1,1) con t-dist (si se puede) y creación de intervalos.
    """
    am = arch_model(resid, mean='Zero', vol='Garch', p=1, q=1, dist='t').fit(disp='off')
    var = am.forecast(horizon=len(future_df), reindex=False).variance.values[-1]
    std = np.sqrt(var)

    nu = am.params.get("nu", None)
    q = t.ppf(1 - (1 - alpha) / 2, nu) if nu else 1.96

    # std puede ser array (por horizonte) o escalar; mantengo la misma lógica que el original
    future_df["upper"] = future_df["pred_trend"] + q * std
    future_df["lower"] = future_df["pred_trend"] - q * std

    future_df["forecast_final"] = future_df["pred_trend"] + seasonal
    future_df["forecast_upper"] = future_df["upper"] + seasonal
    future_df["forecast_lower"] = future_df["lower"] + seasonal
    future_df["volatility"] = std
    return future_df


# =====================================================
# 4. ORQUESTACIÓN (Pipeline maestro)
# =====================================================

def run_forecast_pipeline(D4NN, target_col, json_path, future_days=15):
    """
    Pipeline maestro adaptado: misma salida que primer programa (df_final, fut, model),
    estructura modular al estilo del segundo y STL en la descomposición.
    """
    df = load_and_aggregate(D4NN, target_col)
    df_final = df[[target_col]].copy()
    df_final[target_col + "_FORE"] = df[[target_col]].copy()
    df_final["FORE_LOWER"] = df[[target_col]].copy()
    df_final["FORE_UPPER"] = df[[target_col]].copy()

    df, result = add_seasonal_and_calendar(df, target_col)
    df, BC_json = add_business_features(df, json_path)
    df = add_diff_features(df)

    feature_cols = [
        "diff_lag1", "diff_lag7", "diff_lag30", "diff_ma7", "diff_ma30",
        "sin_dia", "cos_dia", "sin_mes", "cos_mes", "DiaMayorBoletaje",
        "MesMayorBoletaje", "PORCENT_PROMO", "FOP"
    ]

    # --- Train / predict ---
    Xtr, Xts, ytr, yts, split = build_train_test(df, feature_cols)
    model = train_xgboost(Xtr, ytr)

    pred = model.predict(Xts)
    real, recon = reconstruct_trend(df, pred, split)

    # --- Evaluación: RMSE, MAPE, R² sobre niveles del trend reconstruido ---
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
    rmse = np.sqrt(mean_squared_error(real, recon))
    mape = mean_absolute_percentage_error(real, recon) * 100
    r2 = r2_score(real, recon)

    print("\n📊 RESULTADOS XGBoost (Diferencias → Trend reconstruido) — EVALUACIÓN:")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAPE : {mape:.2f}%")
    print(f"R²   : {r2:.6f}")

    resid = real - recon

    # --- Forecast futuro y GARCH ---
    fut = forecast_autoregressive(df, model, BC_json, future_days)
    seas = np.resize(result.seasonal[-14:].values, future_days)
    fut = apply_garch_intervals(resid, fut, seas)

    # ✅ CORRECCIÓN: normalizar índices para evitar nanosegundos raros
    fut.index = fut.index.round("s")
    df_final.index = df_final.index.round("s")

    # ✅ Intersección segura (solo índices que existen en df_final)
    idx = fut.index.intersection(df_final.index)

    # ✅ Asignación sin riesgo de KeyError
    df_final.loc[idx, target_col + "_FORE"] = fut.loc[idx, "forecast_final"]
    df_final.loc[idx, "FORE_UPPER"] = fut.loc[idx, "forecast_upper"]
    df_final.loc[idx, "FORE_LOWER"] = fut.loc[idx, "forecast_lower"]

    df_final = df_final.fillna(0)

    return df_final, fut, model


# ================= EJECUCIÓN =================
if __name__ == "__main__":
    ruta_principal = os.getcwd()
    BuenasCarac_path = os.path.join(ruta_principal, "Files", "BuenaCaracteristicas.json")

    # Llamada de ejemplo (ajusta D4NN en tu entorno antes de ejecutar)
    df_forecast, future_df, model = run_forecast_pipeline(
         D4NN,
         target_col="BOLETOS_VEND",
         json_path=BuenasCarac_path,
         future_days=15
     )

df_forecast


📊 RESULTADOS XGBoost (Diferencias → Trend reconstruido) — EVALUACIÓN:
RMSE : 18.5423
MAPE : 8.76%
R²   : 0.939643


,BOLETOS_VEND,BOLETOS_VEND_FORE,FORE_LOWER,FORE_UPPER
FECHA_OPERACION,,,,
2023-01-01,375.0,375.000000,375.000000,375.000000
2023-01-02,453.0,453.000000,453.000000,453.000000
2023-01-03,501.0,501.000000,501.000000,501.000000
2023-01-04,393.0,393.000000,393.000000,393.000000
2023-01-05,362.0,362.000000,362.000000,362.000000
...,...,...,...,...
2025-11-06,171.0,171.000000,171.000000,171.000000
2025-11-07,142.0,142.000000,142.000000,142.000000
2025-11-08,103.0,103.000000,103.000000,103.000000


In [4]:
"""# =====================================================
# 1. DATA PREPARATION
# =====================================================
def load_and_aggregate(D4NN, target_col):
    Frame = D4NN[['FECHA_CORRIDA', target_col, 'MES', 'PORCENT_PROMO', 'FOP']].copy()
    agg_dict = {col:'mean' for col in Frame.columns if col not in ['FECHA_CORRIDA', target_col]}
    agg_dict[target_col] = 'sum'
    
    df = Frame.groupby('FECHA_CORRIDA').agg(agg_dict)
    df.index = pd.to_datetime(df.index)
    return df.sort_index()


def add_seasonal_and_calendar(df, target_col):
    result = seasonal_decompose(df[target_col], model='additive', period=14)
    df['trend'] = result.trend
    df['DiaSemana'] = df.index.dayofweek

    df["sin_mes"] = np.sin(2*np.pi*df["MES"]/12)
    df["cos_mes"] = np.cos(2*np.pi*df["MES"]/12)
    df["sin_dia"] = np.sin(2*np.pi*df["DiaSemana"]/7)
    df["cos_dia"] = np.cos(2*np.pi*df["DiaSemana"]/7)

    return df, result


def add_business_features(df, json_path):
    with open(json_path, 'r') as f:
        BC = json.load(f)
    df['DiaMayorBoletaje'] = df['DiaSemana'].isin(BC["DiaBueno"])
    df['MesMayorBoletaje'] = df['MES'].isin(BC["MesBueno"])
    return df, BC


def add_diff_features(df):
    df['trend_diff'] = df['trend'].diff()
    df["diff_lag1"]  = df["trend_diff"].shift(1)
    df["diff_lag7"]  = df["trend_diff"].shift(7)
    df["diff_lag30"] = df["trend_diff"].shift(30)

    df["diff_ma7"]  = df["trend_diff"].rolling(7).mean()
    df["diff_ma30"] = df["trend_diff"].rolling(30).mean()
    return df.dropna()


# =====================================================
# 2. TRAINING
# =====================================================
def build_train_test(df, feature_cols, target="trend_diff", test_ratio=0.2):
    X = df[feature_cols]
    y = df[target]
    split = int(len(df)*(1-test_ratio))
    return X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:], split


def train_xgboost(X_train, y_train):
    model = XGBRegressor(
        n_estimators=800, learning_rate=0.03,
        max_depth=6, subsample=0.8, colsample_bytree=0.8
    )
    model.fit(X_train, y_train)
    return model


def reconstruct_trend(df, y_pred, split):
    last = df['trend'].iloc[split-1]
    trend_pred = np.cumsum(y_pred) + last
    trend_real = df["trend"].iloc[split:].values
    return trend_real, trend_pred


# =====================================================
# 3. FUTURE FORECAST
# =====================================================
def forecast_autoregressive(df, model, BC_json, days=15):
    last_date = df.index[-1]
    future = pd.DataFrame(index=pd.date_range(last_date+pd.Timedelta(1), periods=days))

    # calendar
    future["MES"] = future.index.month
    future["DiaSemana"] = future.index.dayofweek
    future["sin_mes"] = np.sin(2*np.pi*future["MES"]/12)
    future["cos_mes"] = np.cos(2*np.pi*future["MES"]/12)
    future["sin_dia"] = np.sin(2*np.pi*future["DiaSemana"]/7)
    future["cos_dia"] = np.cos(2*np.pi*future["DiaSemana"]/7)

    # business rules
    future["DiaMayorBoletaje"] = future["DiaSemana"].isin(BC_json["DiaBueno"])
    future["MesMayorBoletaje"] = future["MES"].isin(BC_json["MesBueno"])

    # exogenas (naive)
    future["PORCENT_PROMO"] = df["PORCENT_PROMO"].iloc[-1]
    future["FOP"] = df["FOP"].iloc[-1]

    # autoregressive loop
    diffs, last_trend, last_diff = [], df["trend"].iloc[-1], df["trend_diff"].iloc[-1]

    for i in range(days):
        row = pd.DataFrame({
            "diff_lag1":[last_diff],
            "diff_lag7":[df["trend_diff"].iloc[-7+i] if i<7 else diffs[i-7]],
            "diff_lag30":[df["trend_diff"].iloc[-30+i] if i<30 else diffs[i-30]],
            "diff_ma7":[df["trend_diff"].iloc[-7:].mean()],
            "diff_ma30":[df["trend_diff"].iloc[-30:].mean()],
            "sin_dia":[future["sin_dia"].iloc[i]],
            "cos_dia":[future["cos_dia"].iloc[i]],
            "sin_mes":[future["sin_mes"].iloc[i]],
            "cos_mes":[future["cos_mes"].iloc[i]],
            "DiaMayorBoletaje":[future["DiaMayorBoletaje"].iloc[i]],
            "MesMayorBoletaje":[future["MesMayorBoletaje"].iloc[i]],
            "PORCENT_PROMO":[future["PORCENT_PROMO"].iloc[i]],
            "FOP":[future["FOP"].iloc[i]],
        })

        pred = model.predict(row)[0]
        diffs.append(pred)
        last_trend += pred
        last_diff = pred

    future["pred_trend"] = np.cumsum(diffs) + df["trend"].iloc[-1]
    return future


def apply_garch_intervals(resid, future_df, seasonal, alpha=0.99):
    am = arch_model(resid, mean='Zero', vol='Garch', p=1, q=1, dist='t').fit(disp='off')
    var = am.forecast(horizon=len(future_df), reindex=False).variance.values[-1]
    std = np.sqrt(var)

    from scipy.stats import t
    nu = am.params.get("nu", None)
    q = t.ppf(1-(1-alpha)/2, nu) if nu else 1.96

    future_df["upper"] = future_df["pred_trend"] + q*std
    future_df["lower"] = future_df["pred_trend"] - q*std

    future_df["forecast_final"] = future_df["pred_trend"] + seasonal
    future_df["forecast_upper"] = future_df["upper"] + seasonal
    future_df["forecast_lower"] = future_df["lower"] + seasonal
    future_df["volatility"] = std
    return future_df


# =====================================================
# 4. ORQUESTACIÓN (Pipeline maestro)
# =====================================================
def run_forecast_pipeline(D4NN, target_col, json_path, future_days=15):

    df = load_and_aggregate(D4NN, target_col)
    df_final = df[[target_col]].copy()
    df_final[target_col+"_FORE"]= df[[target_col]].copy()
    df_final["FORE_LOWER"]= df[[target_col]].copy()
    df_final["FORE_UPPER"]= df[[target_col]].copy()
    
    df, result = add_seasonal_and_calendar(df, target_col)
    df, BC_json = add_business_features(df, json_path)
    df = add_diff_features(df)

    feature_cols = [
      "diff_lag1","diff_lag7","diff_lag30","diff_ma7","diff_ma30",
      "sin_dia","cos_dia","sin_mes","cos_mes","DiaMayorBoletaje",
      "MesMayorBoletaje","PORCENT_PROMO","FOP"
    ]

    # --- Train / predict ---
    Xtr,Xts,ytr,yts,split = build_train_test(df, feature_cols)
    model = train_xgboost(Xtr,ytr)

    pred = model.predict(Xts)
    real, recon = reconstruct_trend(df, pred, split)

    # --- Evaluación: RMSE, MAPE, R² sobre niveles del trend reconstruido ---
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
    rmse = np.sqrt(mean_squared_error(real, recon))
    mape = mean_absolute_percentage_error(real, recon) * 100
    r2   = r2_score(real, recon)

    print("\n📊 RESULTADOS XGBoost (Diferencias → Trend reconstruido) — EVALUACIÓN:")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAPE : {mape:.2f}%")
    print(f"R²   : {r2:.6f}")

    resid = real - recon

    # --- Forecast futuro y GARCH ---
    fut = forecast_autoregressive(df, model, BC_json, future_days)
    seas = np.resize(result.seasonal[-14:].values, future_days)
    fut = apply_garch_intervals(resid, fut, seas)

    # ✅ CORRECCIÓN: normalizar índices para evitar nanosegundos raros
    fut.index = fut.index.round("s")
    df_final.index = df_final.index.round("s")

    # ✅ Intersección segura (solo índices que existen en df_final)
    idx = fut.index.intersection(df_final.index)

    # ✅ Asignación sin riesgo de KeyError
    df_final.loc[idx, target_col+"_FORE"] = fut.loc[idx, "forecast_final"]
    df_final.loc[idx, "FORE_UPPER"] = fut.loc[idx, "forecast_upper"]
    df_final.loc[idx, "FORE_LOWER"] = fut.loc[idx, "forecast_lower"]

    df_final= df_final.fillna(0)

    return df_final, fut, model


ruta_principal = os.getcwd()
BuenasCarac_path = os.path.join(ruta_principal, "Files", "BuenaCaracteristicas.json")

df_forecast, future_df, model = run_forecast_pipeline(
    D4NN,
    target_col="BOLETOS_VEND",
    json_path=BuenasCarac_path,
    future_days=15
)
df_forecast
"""

'# =====================================================\n# 1. DATA PREPARATION\n# =====================================================\ndef load_and_aggregate(D4NN, target_col):\n    Frame = D4NN[[\'FECHA_CORRIDA\', target_col, \'MES\', \'PORCENT_PROMO\', \'FOP\']].copy()\n    agg_dict = {col:\'mean\' for col in Frame.columns if col not in [\'FECHA_CORRIDA\', target_col]}\n    agg_dict[target_col] = \'sum\'\n    \n    df = Frame.groupby(\'FECHA_CORRIDA\').agg(agg_dict)\n    df.index = pd.to_datetime(df.index)\n    return df.sort_index()\n\n\ndef add_seasonal_and_calendar(df, target_col):\n    result = seasonal_decompose(df[target_col], model=\'additive\', period=14)\n    df[\'trend\'] = result.trend\n    df[\'DiaSemana\'] = df.index.dayofweek\n\n    df["sin_mes"] = np.sin(2*np.pi*df["MES"]/12)\n    df["cos_mes"] = np.cos(2*np.pi*df["MES"]/12)\n    df["sin_dia"] = np.sin(2*np.pi*df["DiaSemana"]/7)\n    df["cos_dia"] = np.cos(2*np.pi*df["DiaSemana"]/7)\n\n    return df, result\n\n\ndef